# 7. RANDOM FOREST con variables de mediana en buffer de 180 m

Se incorporando variables de contexto espacial calculadas como **mediana de píxeles dentro de un buffer circular de 180 m** alrededor de cada punto.


Reglas temporales actuales:
   - Landsat 8: regla bimestral con `l8_switch_day`.
   - Sentinel-1 A/B: primera escena posterior dentro de la ventana temporal.
   - Sentinel-1 C: variables generales desde la escena posterior y variables de diferencia desde escena del mismo mes más cercana a la fecha del punto.
2. El buffer se construye en `EPSG:9377`, por tanto el radio está en metros.
3. Las variables `med180` se calculan para todas las bandas actuales de Landsat 8 y Sentinel-1.
4. Dataset A y Dataset C siguen siendo de caso completo; por tanto, si falta cualquier variable puntual o de buffer, el punto se elimina.
5. Dataset B mantiene imputación por mediana de clase, ahora incluyendo también las variables `med180`.
6. La normalización Min-Max se mantiene sin fuga de información: se calcula dentro de cada fold usando solo entrenamiento y luego se aplica al conjunto de prueba.

In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Las variables puntuales representan el valor del píxel que intersecta el punto. Las variables `med180` representan el contexto local alrededor del punto. En términos espaciales, el buffer de 180 m equivale aproximadamente a un entorno de 6 píxeles Landsat/Sentinel-1 de 30 m de radio, aunque el número real de píxeles usados depende de la geometría del buffer, la alineación del raster, NoData y el recorte del AOI.



In [5]:
from pathlib import Path
import logging
import sys

PROJECT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from src.pipeline_rf_Copy1_med180 import (
    run_pipeline,
    run_pipeline_model_c,
    L8_RAW_FEATURES,
    S1_RAW_FEATURES,
    L8_BUFFER_FEATURES,
    S1_BUFFER_FEATURES,
    get_raw_features,
    get_norm_features,
)
from src.config_rf_pipeline_Copy1_med180 import load_config

cfg = load_config(PROJECT / "configs" / "RandomForest_260507_Copy1.json")

log_path = PROJECT / "logs" / "rf_pipeline_260507_Copy1_med180.log"
log_path.parent.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    filename=str(log_path),
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True
)

logger = logging.getLogger("RandomForest_260507_Copy1_med180")
logger.info("Inicio pipeline RF con variables med180 y normalización Min-Max por fold")

print("PROJECT:", PROJECT)
print("Log:", log_path)

PROJECT: C:\Users\Equipo\Tesis\SemestreIV\Objetivo1\Objetivo1_26_02_20
Log: C:\Users\Equipo\Tesis\SemestreIV\Objetivo1\Objetivo1_26_02_20\logs\rf_pipeline_260507_Copy1_med180.log


In [7]:
print("enable_buffer_features:", cfg.enable_buffer_features)
print("buffer_radius_m:", cfg.buffer_radius_m)
print("buffer_statistic:", cfg.buffer_statistic)
print("epsg_work:", cfg.epsg_work)

print("\nBandas Landsat 8 puntuales:")
print(L8_RAW_FEATURES)

print("\nBandas Sentinel-1 puntuales:")
print(S1_RAW_FEATURES)

print("\nVariables Landsat 8 med180:")
print(L8_BUFFER_FEATURES)

print("\nVariables Sentinel-1 med180:")
print(S1_BUFFER_FEATURES)

print("\nVariables crudas activas:")
print(get_raw_features(cfg.enable_buffer_features))

print("\nVariables normalizadas activas:")
print(get_norm_features(cfg.enable_buffer_features))

enable_buffer_features: True
buffer_radius_m: 180.0
buffer_statistic: median
epsg_work: 9377

Bandas Landsat 8 puntuales:
['SR_B5', 'SR_B6', 'SR_B7', 'NDVI', 'EVI', 'NBR', 'NDWI']

Bandas Sentinel-1 puntuales:
['VV', 'VH', 'angle', 'VVVH_ratio', 'VV_Difference', 'VH_Difference', 'VHVV_Difference']

Variables Landsat 8 med180:
['SR_B5_med180', 'SR_B6_med180', 'SR_B7_med180', 'NDVI_med180', 'EVI_med180', 'NBR_med180', 'NDWI_med180']

Variables Sentinel-1 med180:
['VV_med180', 'VH_med180', 'angle_med180', 'VVVH_ratio_med180', 'VV_Difference_med180', 'VH_Difference_med180', 'VHVV_Difference_med180']

Variables crudas activas:
['SR_B5', 'SR_B6', 'SR_B7', 'NDVI', 'EVI', 'NBR', 'NDWI', 'VV', 'VH', 'angle', 'VVVH_ratio', 'VV_Difference', 'VH_Difference', 'VHVV_Difference', 'SR_B5_med180', 'SR_B6_med180', 'SR_B7_med180', 'NDVI_med180', 'EVI_med180', 'NBR_med180', 'NDWI_med180', 'VV_med180', 'VH_med180', 'angle_med180', 'VVVH_ratio_med180', 'VV_Difference_med180', 'VH_Difference_med180', 'VHVV_D

In [9]:
# =========================================================
# MODELOS A Y B
# =========================================================

res = run_pipeline(cfg, logger)

print("========== MODELOS A Y B ==========")
print("OUT_DIR:", res["out_dir"])
print("Raw features:", res["raw_features"])
print("Features normalizadas:", res["features"])
print("Excel Min-Max:", res["normalization_excel"])

print("\nDataset A normalizado:", res["dfA"].shape)
print("Dataset B normalizado:", res["dfB"].shape)

display(res["coverage"])

print("Parámetros finales Modelo A")
display(res["params_A_final"])

print("Parámetros finales Modelo B")
display(res["params_B_final"])

print("Parámetros por fold Modelo A")
display(res["params_A_folds"].head())

print("Parámetros por fold Modelo B")
display(res["params_B_folds"].head())

display(res["foldsA"].head())
display(res["foldsB"].head())
display(res["dfA"].head())
display(res["dfB"].head())

C:\Users\Equipo\Tesis\SemestreIV\Objetivo1\Objetivo1_26_02_20\src\points_loader.py:76: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  g = pd.concat([g_fire, g_abs], ignore_index=True)


=== DEBUG PRE-CV ===
g_points: (1922, 51)
g_master: (1922, 89)
FEATURES: ['SR_B5_norm', 'SR_B6_norm', 'SR_B7_norm', 'NDVI_norm', 'EVI_norm', 'NBR_norm', 'NDWI_norm', 'VV_norm', 'VH_norm', 'angle_norm', 'VVVH_ratio_norm', 'VV_Difference_norm', 'VH_Difference_norm', 'VHVV_Difference_norm', 'SR_B5_med180_norm', 'SR_B6_med180_norm', 'SR_B7_med180_norm', 'NDVI_med180_norm', 'EVI_med180_norm', 'NBR_med180_norm', 'NDWI_med180_norm', 'VV_med180_norm', 'VH_med180_norm', 'angle_med180_norm', 'VVVH_ratio_med180_norm', 'VV_Difference_med180_norm', 'VH_Difference_med180_norm', 'VHVV_Difference_med180_norm']

Conteo label en g_master:
label
1    961
0    961
Name: count, dtype: int64

L8 found:
l8_found
True     1055
False     867
Name: count, dtype: int64

S1 found:
s1_found
False    1292
True      630
Name: count, dtype: int64

NaN por feature cruda en g_master:
SR_B5                     1222
SR_B6                     1222
SR_B7                     1222
NDVI                      1222
EVI          

ValueError: dfA quedó vacío antes de CV. Revisa cobertura completa de FEATURES, disponibilidad L8/S1 y balanceo temporal.